#Experiment 5

#Perceptron vs Multilayer Perceptron with Hyperparameter Tuning

In [15]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dhruvildave/english-handwritten-characters-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'english-handwritten-characters-dataset' dataset.
Path to dataset files: /kaggle/input/english-handwritten-characters-dataset


# Importing Modules

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Pre-Process data

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# Paths
ROOT = "/root/.cache/kagglehub/datasets/dhruvildave/english-handwritten-characters-dataset/versions/3"
CSV_PATH = os.path.join(ROOT, "english.csv")

IMG_SIZE = 28

X = []
y = []

df = pd.read_csv(CSV_PATH)

for _, row in df.iterrows():
    img_path = os.path.join(ROOT, row["image"])

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0          # normalize
    img = img.flatten()        # flatten

    X.append(img)
    y.append(row["label"])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(3410, 784)
(3410,)


# Train-Test-Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Implement PLA from scratch

step function

In [ ]:
def step(z):
    return 1 if z >= 0 else 0

Binary Perceptron

In [ ]:
class Perceptron:
    def __init__(self, lr=1.0, epochs=20):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0

        for _ in range(self.epochs):
            for i in range(n):
                y_hat = step(np.dot(self.w, X[i]) + self.b)
                err = y[i] - y_hat
                self.w += self.lr * err * X[i]
                self.b += self.lr * err

    def score(self, X):
        return X @ self.w + self.b

Multi-class PLA (One-vs-Rest)

In [ ]:
class PLA_OVR:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.models = []

        for c in self.classes:
            y_bin = (y == c).astype(int)
            p = Perceptron()
            p.fit(X, y_bin)
            self.models.append(p)

    def predict(self, X):
        scores = np.column_stack([m.score(X) for m in self.models])
        return self.classes[np.argmax(scores, axis=1)]

In [ ]:
model = PLA_OVR()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro")
rec = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print("Accuracy (%):", acc * 100)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)

Accuracy (%): 15.982404692082111
Precision: 0.18326609606194927
Recall: 0.17481866252027542
F1-score: 0.13545676003462792


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Implement MLP

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_enc = le.fit_transform(y)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),  # hidden layers & neurons
    activation="relu",              # activation function
    solver="adam",                  # backpropagation
    learning_rate_init=0.001,
    batch_size=64,
    max_iter=40,
    early_stopping=True,
    random_state=42
)

mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

acc = accuracy_score(y_test, y_pred) * 100
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("Accuracy (%):", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)


Accuracy (%): 25.95307917888563
Precision: 0.2681668241092929
Recall: 0.2595307917888563
F1-score: 0.23743487096056717


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


# Hyperparameter Tuning

In [13]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

learning_rates = [0.01, 0.001, 0.0005]
batch_sizes = [32, 64]
optimizers = ["sgd", "adam"]
activations = ["relu", "tanh"]

results = []

best_acc = -1
best_cfg = None

for act in activations:
    for opt in optimizers:
        for lr in learning_rates:
            for bs in batch_sizes:
                mlp = MLPClassifier(
                    hidden_layer_sizes=(256, 128),
                    activation=act,
                    solver=opt,
                    learning_rate_init=lr,
                    batch_size=bs,
                    max_iter=40,
                    early_stopping=True,
                    random_state=42
                )
                mlp.fit(X_train, y_train)
                y_pred = mlp.predict(X_test)
                acc = accuracy_score(y_test, y_pred) * 100

                results.append((act, opt, lr, bs, acc))

                if acc > best_acc:
                    best_acc = acc
                    best_cfg = (act, opt, lr, bs)

for act, opt, lr, bs, acc in results:
    print("Activation:", act,
          "Optimizer:", opt,
          "Learning rate:", lr,
          "Batch size:", bs,
          "Accuracy (%):", acc)

print("Best configuration:", best_cfg)
print("Best accuracy (%):", best_acc)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py

Activation: relu Optimizer: sgd Learning rate: 0.01 Batch size: 32 Accuracy (%): 1.6129032258064515
Activation: relu Optimizer: sgd Learning rate: 0.01 Batch size: 64 Accuracy (%): 32.25806451612903
Activation: relu Optimizer: sgd Learning rate: 0.001 Batch size: 32 Accuracy (%): 27.126099706744867
Activation: relu Optimizer: sgd Learning rate: 0.001 Batch size: 64 Accuracy (%): 15.102639296187684
Activation: relu Optimizer: sgd Learning rate: 0.0005 Batch size: 32 Accuracy (%): 10.410557184750733
Activation: relu Optimizer: sgd Learning rate: 0.0005 Batch size: 64 Accuracy (%): 10.410557184750733
Activation: relu Optimizer: adam Learning rate: 0.01 Batch size: 32 Accuracy (%): 1.6129032258064515
Activation: relu Optimizer: adam Learning rate: 0.01 Batch size: 64 Accuracy (%): 1.6129032258064515
Activation: relu Optimizer: adam Learning rate: 0.001 Batch size: 32 Accuracy (%): 31.524926686217007
Activation: relu Optimizer: adam Learning rate: 0.001 Batch size: 64 Accuracy (%): 25.95307

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


In [14]:
pla = PLA_OVR()
pla.fit(X_train, y_train)

y_train_pred_pla = pla.predict(X_train)
pla_train_loss = 1 - accuracy_score(y_train, y_train_pred_pla)

print("Model: PLA")
print("Epochs:", pla.models[0].epochs)
print("Final Training Loss:", pla_train_loss)
print("Convergence Behavior:", "Non-convergent")
print()

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    batch_size=64,
    max_iter=40,
    early_stopping=True,
    random_state=42
)

mlp.fit(X_train, y_train)

print("Model: MLP (Tuned)")
print("Epochs:", mlp.n_iter_)
print("Final Training Loss:", mlp.loss_)
print("Convergence Behavior:", "Stable convergence")

Model: PLA
Epochs: 20
Final Training Loss: 0.7965542521994134
Convergence Behavior: Non-convergent

Model: MLP (Tuned)
Epochs: 40
Final Training Loss: 2.5611872905137925
Convergence Behavior: Stable convergence


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
